# Preprocesamiento y prevención de fuga

Este cuaderno prepara las variables predictoras del conjunto `Regression of Used Car Prices` (Playground Series S4E9). El objetivo es `price`; las transformaciones se encapsulan en `fase-1/features.py` para poder reutilizarlas al entrenar y cargar modelos con `joblib`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

RANDOM_STATE = 42

# El notebook vive en fase-1/notebooks; features.py está un nivel arriba.
FASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "fase-1"
if str(FASE_DIR) not in sys.path:
    sys.path.insert(0, str(FASE_DIR))

from features import (
    build_preprocessor,
    extract_engine_features,
    extract_transmission_features,
)

DATA_PATH = FASE_DIR / "data" / "train.csv"
df = pd.read_csv(DATA_PATH)

print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
print(f"RANDOM_STATE: {RANDOM_STATE}")
df.head()

## Datos faltantes

Primero cuantificamos los nulos, sin eliminar filas de forma automática. Eliminar una fila por cualquier predictor ausente puede descartar muchas observaciones (y sesgar la muestra si la ausencia no es aleatoria). En este conjunto, `fuel_type`, `accident` y `clean_title` son campos categóricos: conservaremos las observaciones e imputaremos una categoría explícita `Desconocido` dentro del pipeline. Así se distingue la falta de registro de una categoría observada y se evita calcular valores usando el test. Los nulos de `engine` o `transmission` también son admisibles: las extracciones quedan en NaN y se imputan en el paso numérico o categórico correspondiente.

In [ ]:
missing = df.isna().sum().rename("nulos")
missing_summary = pd.DataFrame(
    {
        "nulos": missing,
        "porcentaje": (missing / len(df) * 100).round(2),
    }
).sort_values("nulos", ascending=False)
missing_summary

## Partición train/test

Usamos una partición aleatoria 80/20 con semilla fija para reservar una evaluación final reproducible y mantener suficientes filas en train para aprender imputación y categorías. Separamos `price` como objetivo y descartamos `id` antes del split porque es un identificador de fila, no una característica del vehículo. El split aleatorio no revela por sí mismo el objetivo ni usa `id`; sin VIN o identificador de vehículo no podemos hacer una partición agrupada. Registros de vehículos repetidos o muy similares podrían quedar en ambos lados y hacer la evaluación optimista; revisaremos duplicados exactos de predictores, aunque no prueban que sea el mismo vehículo.

In [ ]:
X = df.drop(columns=["price", "id"])
y = df["price"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")
print(f"Duplicados exactos de predictores (antes del split): {X.duplicated().sum():,}")

## Extracción y preprocesamiento

La extracción de HP, litros, cilindros y velocidades es determinista y por fila. Encadenamos las funciones del módulo como `FunctionTransformer` y después el `ColumnTransformer`; por tanto, el objeto completo se puede guardar con `joblib` y reutilizar en inferencia. Las numéricas usan mediana. Para `fuel_type`, tipo de transmisión, accidente y título limpio usamos una categoría explícita de desconocido y one-hot; `brand`, `model` y colores usan `OrdinalEncoder` con código `-1` para categorías no vistas. Esta decisión controla el tamaño que produciría un one-hot de alta cardinalidad; la codificación ordinal puede ser poco apropiada para modelos lineales. El escalado estándar es útil para modelos lineales y basados en distancias, pero puede omitirse si el estimador final es un árbol.

In [ ]:
preprocessing_pipeline = Pipeline(
    steps=[
        ("engine_features", FunctionTransformer(extract_engine_features, validate=False)),
        (
            "transmission_features",
            FunctionTransformer(extract_transmission_features, validate=False),
        ),
        ("preprocessor", build_preprocessor()),
    ]
)

# El pipeline aprende imputadores, categorías y escalas exclusivamente de X_train.
preprocessing_pipeline.fit(X_train)
X_train_processed = preprocessing_pipeline.transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print(f"X_train antes/después: {X_train.shape} -> {X_train_processed.shape}")
print(f"X_test antes/después:  {X_test.shape} -> {X_test_processed.shape}")

## Outliers y escala del objetivo

No eliminaremos automáticamente outliers de `price` ni de predictores: autos de lujo, antiguos o con alto kilometraje pueden ser observaciones válidas. El resumen por cuantiles y la regla IQR siguiente cuantifican extremos, pero no deciden borrarlos; cualquier recorte tendría que justificarse con conocimiento del dominio y ajustarse usando solo train. Por ahora mantenemos `price` en su escala original para que las métricas sean interpretables en moneda. `log1p(price)` puede reducir asimetría y ser útil si se optimiza error relativo, pero cambia el objetivo y exige invertir con `expm1` (idealmente con `TransformedTargetRegressor`); se comparará en validación cruzada, no sobre este test final.

In [ ]:
price_quantiles = y_train.quantile([0.00, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00])
q1, q3 = y_train.quantile([0.25, 0.75])
iqr = q3 - q1
outlier_count = ((y_train < q1 - 1.5 * iqr) | (y_train > q3 + 1.5 * iqr)).sum()

print("Cuantiles de price en train:")
display(price_quantiles)
print(f"Observaciones train fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]: {outlier_count:,}")
print(f"Asimetría price en train: {y_train.skew():.3f}")
print(f"Asimetría log1p(price) en train: {np.log1p(y_train).skew():.3f}")

## Prevención de fuga de información

- `price` se separa como objetivo y no forma parte de `X`.
- `id` se descarta antes de dividir, por lo que no puede actuar como predictor accidental.
- El pipeline se ajusta una sola vez con `X_train`; `X_test` solo recibe `transform`.
- Imputación, categorías de los codificadores y parámetros del escalador viven dentro del pipeline y aprenden durante ese mismo `fit` de train.
- Los predictores disponibles son atributos del vehículo y reportes de condición/título. No hay campos explícitos de venta posterior; antes de desplegar, hay que confirmar que `accident` y `clean_title` estén disponibles en el instante real de predicción.
- También se revisaron duplicados exactos de predictores. Sin VIN no es posible probar ni agrupar vehículos repetidos; el split aleatorio podría ser optimista si el mismo vehículo aparece más de una vez. No se usan precios de test para diseñar las transformaciones.

In [ ]:
assert "price" not in X_train.columns and "price" not in X_test.columns
assert "id" not in X_train.columns and "id" not in X_test.columns
assert X_train.index.intersection(X_test.index).empty
assert "sale_date" not in X.columns and "final_sale_price" not in X.columns

fitted_preprocessor = preprocessing_pipeline.named_steps["preprocessor"]
numeric_imputer = fitted_preprocessor.named_transformers_["numeric"].named_steps["imputer"]
numeric_scaler = fitted_preprocessor.named_transformers_["numeric"].named_steps["scaler"]
ordinal_encoder = fitted_preprocessor.named_transformers_["ordinal"].named_steps["encoder"]
onehot_encoder = fitted_preprocessor.named_transformers_["onehot"].named_steps["encoder"]

assert len(numeric_imputer.statistics_) == 6
assert len(numeric_scaler.mean_) == 6
assert len(ordinal_encoder.categories_) == 4
assert len(onehot_encoder.categories_) == 4
assert X_train_processed.shape[0] == len(X_train)
assert X_test_processed.shape[0] == len(X_test)

print("Controles de fuga superados: objetivo/ID excluidos, índices disjuntos y")
print("transformadores ajustados mediante preprocessing_pipeline.fit(X_train).")